<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/00_baselines/score_ocm_cloudsen12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet "omnicloudmask[legacy]==1.7.0" tacoreader transformers scipy

In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import pandas as pd
import torch
from IPython.display import display

from cloudband.colab.session import reload_package, start

reload_package("cloudband")

from cloudband.baselines import ocm
from cloudband.eval.palette import assign_colors
from cloudband.eval.plots import plot_experiment_comparison
from cloudband.eval.report import compare
from cloudband.pipelines import phase0, phase2_eval
from cloudband.train.store import RunStore, ensure_shared_folder

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

BASELINE_DIR = Path("/content/drive/MyDrive/cloudband/results")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FOLDER = Path("/content/drive/MyDrive/cloudband_treino")
ensure_shared_folder(DRIVE_FOLDER)
RESULTS_DIR = DRIVE_FOLDER / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

session = start(PROJECT_DIR, require_accelerator=True)
store = RunStore(
    results_dir=RESULTS_DIR,
    checkpoint_dir=DRIVE_FOLDER / "checkpoints",
    source_dir=Path(PROJECT_DIR) / "models",
)
store.wake()
print(f"published OCM results: {BASELINE_DIR}")
print(f"phase 2 shared folder: {DRIVE_FOLDER}")

In [ ]:
import getpass
import os

import requests

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Hugging Face token (type 'Read' token, hidden): ").strip()

os.environ.pop("HF_TOKEN", None)
if token:
    reply = requests.get(
        "https://huggingface.co/api/whoami-v2",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30,
    )
    if reply.status_code == 200:
        os.environ["HF_TOKEN"] = token
        print(f"token accepted for the account {reply.json().get('name', '?')}")
    else:
        print(f"token refused (HTTP {reply.status_code}); going on without it")
else:
    print("no token: requests use the shared anonymous quota")
del token

In [ ]:
tables = phase2_eval.load_test_tables()
test_table = tables["test"]
print("test patches, L1C (primary):", len(tables["test"]))
print("test patches, L2A (secondary):", len(tables["test_l2a"]))

## OmniCloudMask publicado, por versão do modelo

In [ ]:
ocm.check_version()

MODEL_VERSIONS = (ocm.PAPER_MODEL_VERSION, 2.0, 3.0, ocm.LATEST_MODEL_VERSION)
print("package:", ocm.package_version())
print("scoring model versions:", MODEL_VERSIONS)

In [ ]:
scores, paper_tables = phase0.score_published_versions(
    test_table,
    MODEL_VERSIONS,
    BASELINE_DIR,
    package_versions=session.package_versions,
    progress=phase0.progress_printer(),
)
pd.concat(scores, names=["model_version", "experiment"])

In [ ]:
pd.concat(paper_tables, names=["model_version", "experiment"])

In [ ]:
median_tables = {phase0.version_label(v): table for v, table in paper_tables.items()}
comparison = compare(median_tables, column="median_boa")
colors = assign_colors(list(comparison.columns))
comparison

In [ ]:
version_figures = plot_experiment_comparison(
    comparison, colors, metric_label="Median BOA (%)"
)
for experiment, figure in version_figures.items():
    figure.savefig(
        BASELINE_DIR / f"ocm_versions_{experiment}_median_boa.png",
        dpi=150,
        bbox_inches="tight",
    )
    display(figure)

## Fase 2: OCM retreinado × Swin-T + UPerNet, com o OCM publicado 4.0 como referência

In [ ]:
cache = phase2_eval.build_test_cache(Path("/content/cloudband_cache"), tables)

In [ ]:
published = phase2_eval.score_published(cache, tables, RESULTS_DIR)

gap = phase2_eval.published_gap(published, BASELINE_DIR)
if gap is not None:
    print(f"largest gap to the per-patch BOA of the section above: {gap:.2e}")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

counts = phase2_eval.score_trained(cache, tables, store, RESULTS_DIR, DEVICE)

gaps = phase2_eval.legacy_gaps(counts["test"], RESULTS_DIR)
if not gaps.empty:
    display(gaps)

In [ ]:
combined = phase2_eval.with_published(counts, published)
reports = phase2_eval.build_reports(combined)
phase2_eval.show_reports(reports, display)

agreement = phase2_eval.level_agreement(reports)
if agreement is not None:
    print("\nOCM minus Swin per ROI has the same sign at L1C and L2A:", agreement)

In [ ]:
saved = phase2_eval.save_reports(reports, RESULTS_DIR)
print(f"{len(saved)} tables saved to {RESULTS_DIR}" if saved else "nothing to save yet")

In [ ]:
figures = phase2_eval.build_figures(reports, combined)
saved = phase2_eval.save_figures(figures, FIGURES_DIR)
for figure in figures.values():
    display(figure)
print(f"{len(saved)} figures saved to {FIGURES_DIR}" if saved else "nothing to plot yet")